# 09 — Grupimi dhe zbulimi i strukturës

## Konteksti dhe problemi shkencor

Të dhënat atmosferike përdoren për të treguar se clustering zbulon strukturë sipas një kriteri, jo ``etiketa të vërteta automatikisht.

### Objektivat e të nxënit

- të implementoni k-means;
- të studioni inicializimin dhe K;
- të përdorni silhouette dhe dendrogram;
- të njihni rastet ku k-means dështon.

> **Parimi i laboratorit:** matematikë → implementim nga parimet e para → implementim me bibliotekë → krahasim numerik → interpretim fizik.


In [ ]:
from pathlib import Path
import sys, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_validate, StratifiedKFold, KFold, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, LogisticRegression, Ridge, Lasso, Perceptron, SGDRegressor
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
DATA = Path("../datasets/synthetic")
SRC = Path("../src").resolve()
if str(SRC.parent) not in sys.path:
    sys.path.insert(0, str(SRC.parent))
np.set_printoptions(precision=4, suppress=True)
RANDOM_STATE = 42


## 1. Regjime atmosferike pa etiketa

In [ ]:
df=pd.read_csv(DATA/'atmospheric_regimes.csv'); X=df[['temperatura_C','RH_pct','presioni_hPa']].to_numpy(); plt.scatter(X[:,0],X[:,1],s=14); plt.xlabel('T'); plt.ylabel('RH'); plt.show()

## 2. k-means manual

In [ ]:
from src.clustering.manual import kmeans
Xs=StandardScaler().fit_transform(X); labels,centers,history=kmeans(Xs,3,seed=0); print(centers)

In [ ]:
plt.scatter(X[:,0],X[:,1],c=labels,s=14); plt.show()

## 3. Trajektorja e qendrave

In [ ]:
H=np.array(history); plt.scatter(Xs[:,0],Xs[:,1],c=labels,s=8,alpha=.25);
for j in range(3): plt.plot(H[:,j,0],H[:,j,1],marker='o'); plt.show()

## 4. Krahasim me KMeans dhe k-means++

In [ ]:
from sklearn.cluster import KMeans
for seed in range(5):
 m=KMeans(n_clusters=3,n_init=1,init='random',random_state=seed).fit(Xs); print(seed,m.inertia_)
print('kmeans++',KMeans(n_clusters=3,n_init=20,random_state=0).fit(Xs).inertia_)

## 5. Zgjedhja e K

In [ ]:
from sklearn.metrics import silhouette_score
rows=[]
for k in range(2,8):
 m=KMeans(k,n_init=20,random_state=0).fit(Xs); rows.append((k,m.inertia_,silhouette_score(Xs,m.labels_)))
r=pd.DataFrame(rows,columns=['k','inertia','silhouette']); r

In [ ]:
fig,ax=plt.subplots(); ax.plot(r.k,r.inertia,marker='o'); ax.set_xlabel('k'); ax.set_ylabel('inertia'); plt.show(); plt.plot(r.k,r.silhouette,marker='o'); plt.xlabel('k'); plt.ylabel('silhouette'); plt.show()

## 6. Grupimi hierarkik

In [ ]:
from scipy.cluster.hierarchy import linkage,dendrogram
Z=linkage(Xs[:120],method='ward'); plt.figure(figsize=(10,4)); dendrogram(Z,no_labels=True); plt.show()

## 7. Rast dështimi: forma jo-sferike

In [ ]:
from sklearn.datasets import make_moons
Xm,ym=make_moons(500,noise=.06,random_state=0); km=KMeans(2,n_init=20,random_state=0).fit(Xm); plt.scatter(Xm[:,0],Xm[:,1],c=km.labels_,s=12); plt.title('k-means mbi gjysmëhëna'); plt.show()

## 8. Standardizimi si zgjedhje modeli

In [ ]:
m1=KMeans(3,n_init=20,random_state=0).fit(X); m2=KMeans(3,n_init=20,random_state=0).fit(Xs); print('silhouette raw=',silhouette_score(StandardScaler().fit_transform(X),m1.labels_),'scaled=',silhouette_score(Xs,m2.labels_))

## 9. Ushtrime

- Provoni 20 inicializime manuale.
- Krahasoni `single`, `complete`, `average`, `ward`.
- Hiqni presionin dhe shihni nëse grupet mbeten.
- Jepni një argument fizik pse tre cluster-e mund të jenë artefakt.

## 10. Shembull i plotë

Ndërtoni një raport të grupimit pa përdorur etiketën e fshehtë: zgjedhni k, analizoni stabilitetin dhe përshkruani çdo grup në njësitë fizike origjinale.

## 11. Playground

Vetëm në fund përdorni kolonën `regjim_i_fshehtë` për të parë sa përputhet struktura numerike me mekanizmin që gjeneroi të dhënat. Diskutoni pse kjo nuk do të ishte e disponueshme në një problem real unsupervised.